# 3. Generate validation predictions

**Inputs:** trained component models, selected features and validation features.

**Output:** one Parquet file containing the eight internal component predictions for every horizon.

In [ ]:
# Step 1 - Imports and component models

import gc
import os
import sys

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

sys.path.append(os.path.abspath("../../../")) ; from EPF import variables
sys.path.append(os.path.abspath("../../2_Features_build")) ; import target_features

HORIZONS = range(1, variables.HORIZON_COUNT+1)
COMPONENT_MODELS = [
    ("base_l1", "normal", "full_range_regressor_clipped_MAE_loss", "regression"),
    ("base_l2", "arcsinh", "full_range_regressor_unclipped_RMSE_loss", "regression"),
    ("spike_probability", "spikes", "positive_spike_classifier_binary_loss", "probability"),
    ("spike_mae", "spikes", "positive_spike_regressor_unclipped_mae_loss", "regression"),
    ("spike_q90", "spikes", "positive_spike_regressor_unclipped_quantile_loss", "regression"),
    ("dip_probability", "dips", "negative_spike_classifer_unclipped_binary_loss", "probability"),
    ("dip_mae", "dips", "negative_spike_regressor_unclipped_mae_loss", "regression"),
    ("dip_q10", "dips", "negative_spike_regressor_unclipped_quantile_loss", "regression"),
]

In [ ]:
# Step 2 - Load validation data

def read_period(path, columns):
    index_col = (pq.ParquetFile(path).schema_arrow.pandas_metadata or {})["index_columns"][0]
    return pd.read_parquet(path, columns=columns, filters=[(index_col, ">", variables.VALID_START), (index_col, "<=", variables.TEST_START)]).sort_index().astype(np.float32)

objectives = list(dict.fromkeys(model[1] for model in COMPONENT_MODELS))
selected_by_objective = {objective: pd.read_parquet(variables.SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet") for objective in objectives}

def selected_features(objective, horizon):
    selected = selected_by_objective[objective]
    return selected.loc[selected[f"h{horizon}"].astype(bool), "feature"].tolist()

validation_index = read_period(variables.AGG_TARGET_DATASET_PATH, ["target_h1"]).index

In [ ]:
# Step 3 - Predict and save

predictions = {}

for objective in objectives:
    objective_features = sorted({feature for horizon in HORIZONS for feature in selected_features(objective, horizon)})
    features = read_period(variables.FEATURES_DATASET_PATH, objective_features).reindex(validation_index)
    models = [model for model in COMPONENT_MODELS if model[1] == objective]

    for horizon in tqdm(HORIZONS, desc=objective):
        model_input = target_features.append_target_time_feats(features[selected_features(objective, horizon)], horizon)
        for output_name, _, model_name, prediction_type in models:
            model = joblib.load(variables.TRAINED_MODELS_PATH / f"h{horizon:02d}_{model_name}.joblib")
            prediction = model.predict_proba(model_input)[:, 1] if prediction_type == "probability" else np.sinh(model.predict(model_input))*variables.PRICE_TRANSFORM_SCALE
            predictions[f"{output_name}_h{horizon}"] = np.asarray(prediction, dtype=np.float32)
            del model
        del model_input
        gc.collect()

    del features
    gc.collect()

predictions = pd.DataFrame(predictions, index=validation_index)
predictions.index.name = "date"
predictions.to_parquet(variables.VALIDATION_COMPONENT_PREDICTIONS_PATH)

display(predictions.head())
print(predictions.shape)
print(variables.VALIDATION_COMPONENT_PREDICTIONS_PATH)